**STAT 453: Introduction to Deep Learning and Generative Models (Fall 2026)**

Instructor: Ben Lengerich


Course website: https://adaptinfer.org/dgm-fall-2026/

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdaptInfer/dgm-fall-2026/blob/main/assets/hw/hw4/STAT453_hw04.ipynb)

## How to Submit

1. **Run every cell top to bottom before exporting**, so your output isn't stale:
   - Jupyter: `Kernel → Restart & Run All`
   - Colab: `Runtime → Restart and run all`
2. **Export this notebook to PDF:**
   - Colab: `File → Print → Save as PDF`
   - Local Jupyter: `File → Save and Export Notebook As → PDF`, or from a terminal: `jupyter nbconvert --to webpdf --allow-chromium-download STAT453_hw04.ipynb`
3. **Check the PDF** — make sure plots aren't cut off at page edges and every cell that should have output actually shows it.
4. **Name the file `<netid>_hw4.pdf`** (e.g. `lengerich_hw4.pdf`).
5. **Submit the PDF on Canvas** by the deadline listed on the [homework page](https://adaptinfer.org/dgm-fall-2026/homework/).

# HW 4: Deep Generative Models — VAEs, Diffusion, and GANs (70 pts)

A classifier learns $P(Y \mid X)$. This assignment is about networks that learn $P(X)$ well enough to **sample new data**. It covers the three main families of deep generative models:

- **Part A (25 pts) — Variational autoencoders.** Implement the reparameterization trick, the closed-form KL term, and the VAE loss, each checked against `torch.distributions`. Train a VAE with a 2D latent space on MNIST and see what goes wrong when you switch the KL term off and try to *sample*. Then measure how tight the ELBO actually is with importance sampling.
- **Part B (30 pts) — Diffusion models.** Implement the noise schedule and the closed-form forward (noising) process, the training loss, and the DDPM sampling loop. Train a denoiser on a 2D toy distribution and watch noise turn back into data. Finally, turn the denoiser's noise prediction into an estimate of the clean data point and see the vector field it defines.
- **Part C (15 pts) — GANs.** Implement the discriminator loss and two generator losses (the original minimax loss and the "non-saturating" one). Implement one alternating GAN training step, see why one generator loss stops learning when the discriminator gets ahead, and compare a GAN's samples with diffusion's on the same data.

**How it's graded.** Everything you hand in is code. Cells marked `# <your code>` are the parts you fill in, and each one is followed by a self-check cell built from `assert` statements. If it runs without an `AssertionError` and prints `looks good!`, you're on the right track (passing the self-checks does not guarantee full credit). Cells marked **No modification required** run experiments for you to look at; they aren't graded.

Everything runs on a CPU in a few minutes. A GPU is not needed.

### Important!

**Due Friday, November 13 at 11:59 PM. The homework assignment should be submitted via Canvas.**

See the submission checklist above. If you need help, come to office hours (times are on the [logistics page](https://adaptinfer.org/dgm-fall-2026/logistics/)) or contact the TA or the instructor.

## 0) Imports and data

**No modification required.** Run these cells. The first one downloads MNIST (Part A). The second defines the 2D toy distribution used in Parts B and C: a ring of 8 tight Gaussian clusters. A good generative model should produce points in **all 8** clusters (*mode coverage*), and **only** near the clusters (*sample quality*). `ring_stats()` measures both.

In [ ]:
import math
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
%matplotlib inline

torch.manual_seed(453)

mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True)
mnist_test = torchvision.datasets.MNIST(root='./data', train=False, download=True)
perm = torch.randperm(len(mnist_train), generator=torch.Generator().manual_seed(453))
X_mnist = mnist_train.data[perm[:20000]].float().view(-1, 784) / 255.     # pixel intensities in [0, 1]
X_mnist_test = mnist_test.data[:2000].float().view(-1, 784) / 255.
y_mnist_test = mnist_test.targets[:2000]
print('MNIST train', tuple(X_mnist.shape), '| test', tuple(X_mnist_test.shape))

In [ ]:
RING_K, RING_R, RING_S = 8, 2.0, 0.05
RING_CENTERS = torch.stack([RING_R * torch.cos(2 * math.pi * torch.arange(RING_K) / RING_K),
                            RING_R * torch.sin(2 * math.pi * torch.arange(RING_K) / RING_K)], dim=1)

def sample_ring(n, generator=None):
    which = torch.randint(0, RING_K, (n,), generator=generator)
    return RING_CENTERS[which] + RING_S * torch.randn(n, 2, generator=generator)

def ring_stats(x, threshold=0.2):
    # A sample is "good" if it lands within `threshold` of a cluster center (4 cluster std devs).
    # A mode is "covered" if it gets at least 1% of the samples it would get under the true distribution.
    dist, nearest = torch.cdist(x, RING_CENTERS).min(dim=1)
    good = dist < threshold
    counts = torch.bincount(nearest[good], minlength=RING_K)
    return {'modes_covered': int((counts > 0.01 * len(x) / RING_K).sum()), 'frac_good': good.float().mean().item()}

def plot_points(ax, x, title, color='C0'):
    ax.scatter(x[:, 0], x[:, 1], s=2, alpha=0.4, color=color)
    ax.set(xlim=(-3.2, 3.2), ylim=(-3.2, 3.2), aspect='equal', title=title)
    ax.set_xticks([]); ax.set_yticks([])

X_ring = sample_ring(20000, generator=torch.Generator().manual_seed(453))
fig, ax = plt.subplots(figsize=(3.5, 3.5))
plot_points(ax, X_ring[:3000], 'Ring of 8 Gaussians (data)')
plt.show()
print('ring_stats on real data:', ring_stats(X_ring[:3000]))

## Part A: Variational Autoencoders (25 pts)

A VAE has an **encoder** $q_\phi(z \mid x) = \mathcal{N}\big(\mu_\phi(x), \operatorname{diag}(\sigma^2_\phi(x))\big)$, a **decoder** $p_\theta(x \mid z)$, and a **prior** $p(z) = \mathcal{N}(0, I)$. It is trained by maximizing the evidence lower bound (ELBO):

$$\log p_\theta(x) \;\ge\; \underbrace{\mathbb{E}_{q_\phi(z\mid x)}\big[\log p_\theta(x \mid z)\big]}_{\text{reconstruction}} \;-\; \underbrace{\mathrm{KL}\big(q_\phi(z \mid x)\,\big\|\,p(z)\big)}_{\text{regularizer}}$$

Following common practice, the encoder outputs $\log \sigma^2$ (`logvar`) rather than $\sigma$, so it never has to produce a positive number directly.

### A.1) Reparameterization and the KL term (9 pts)

Complete:
- `reparameterize(mu, logvar)`: return a sample $z = \mu + \sigma \odot \varepsilon$ with $\varepsilon \sim \mathcal{N}(0, I)$, where $\sigma = \exp(\tfrac12 \log\sigma^2)$. Writing the sample this way (the *reparameterization trick*) makes $z$ a differentiable function of $\mu$ and $\sigma$, so gradients can flow back into the encoder.
- `kl_to_standard_normal(mu, logvar)`: the KL divergence from the encoder's diagonal Gaussian to the prior, which has a closed form:
$$\mathrm{KL}\big(\mathcal{N}(\mu, \operatorname{diag}(\sigma^2))\,\big\|\,\mathcal{N}(0, I)\big) = \sum_{j} \tfrac12\left(\mu_j^2 + \sigma_j^2 - 1 - \log\sigma_j^2\right).$$
Sum **over latent dimensions**, returning one number per example, shape `(batch,)`.

In [ ]:
def reparameterize(mu, logvar):
    std = # <your code>
    eps = # <your code>
    return # <your code>

def kl_to_standard_normal(mu, logvar):
    return # <your code>

In [ ]:
from torch.distributions import Normal, Bernoulli, kl_divergence

torch.manual_seed(0)
mu, logvar = torch.randn(5, 3), torch.randn(5, 3)
ref = kl_divergence(Normal(mu, torch.exp(0.5 * logvar)), Normal(0.0, 1.0)).sum(dim=1)
assert kl_to_standard_normal(mu, logvar).shape == (5,), "KL should return one value per example"
assert torch.allclose(kl_to_standard_normal(mu, logvar), ref, atol=1e-5), "KL doesn't match torch.distributions"

mu = torch.tensor([1.0, -2.0], requires_grad=True)
logvar = torch.log(torch.tensor([4.0, 0.25])).requires_grad_(True)
z = reparameterize(mu.expand(200000, 2), logvar.expand(200000, 2))
assert torch.allclose(z.mean(dim=0), mu.detach(), atol=0.03), "samples have the wrong mean"
assert torch.allclose(z.std(dim=0), torch.tensor([2.0, 0.5]), rtol=0.02), "samples have the wrong std"
z.sum().backward()
assert mu.grad is not None and logvar.grad is not None and logvar.grad.abs().sum() > 0, \
    "gradients should flow back to mu and logvar"
print('looks good!')

### A.2) The VAE loss (7 pts)

We treat each pixel as a Bernoulli variable whose probability is set by the decoder's output **logit**, so $-\log p_\theta(x \mid z)$ is the binary cross-entropy **summed over the 784 pixels**. Complete `vae_loss()` so it returns the negative ELBO averaged over the batch, using one sample of $z$ per example:

$$\text{loss} = \frac{1}{B}\sum_{i=1}^{B}\Big[\underbrace{\textstyle\sum_{j=1}^{784} \mathrm{BCE}(\text{logit}_{ij}, x_{ij})}_{\texttt{rec}} \;+\; \beta\cdot \underbrace{\mathrm{KL}\big(q(z\mid x_i)\,\|\,p(z)\big)}_{\texttt{kl}}\Big]$$

$\beta = 1$ is the true ELBO. We'll vary $\beta$ in A.3. *(Hint: `F.binary_cross_entropy_with_logits(..., reduction='none')`.)* Return `rec` and `kl` as batch averages as well, for logging.

In [ ]:
def vae_loss(logits, x, mu, logvar, beta=1.0):
    rec = # <your code>
    kl = # <your code>
    return rec + beta * kl, rec, kl

In [ ]:
torch.manual_seed(0)
logits, x = torch.randn(4, 784), torch.rand(4, 784).round()
mu, logvar = torch.randn(4, 2), torch.randn(4, 2)
loss, rec, kl = vae_loss(logits, x, mu, logvar, beta=2.0)
ref_rec = -Bernoulli(logits=logits).log_prob(x).sum(dim=1).mean()
ref_kl = kl_divergence(Normal(mu, torch.exp(0.5 * logvar)), Normal(0.0, 1.0)).sum(dim=1).mean()
assert torch.allclose(rec, ref_rec, atol=1e-3), "rec should be BCE summed over pixels, averaged over the batch"
assert torch.allclose(kl, ref_kl, atol=1e-4), "kl should be averaged over the batch"
assert torch.allclose(loss, ref_rec + 2.0 * ref_kl, atol=1e-3), "loss should be rec + beta * kl"
print('looks good!')

### A.3) Train two VAEs: with and without the KL term

**No modification required.** This trains two VAEs on 20,000 MNIST digits with a **2D** latent space (so we can plot it): one with $\beta = 1$ (the real ELBO) and one with $\beta = 0$ (the KL term switched off). It uses *your* `reparameterize` and `vae_loss`. For each model it shows:

1. the encoder means $\mu(x)$ of 2,000 test digits, colored by digit label, with the prior's $\pm 2\sigma$ circle drawn in black;
2. reconstructions of test digits (top: original, bottom: reconstruction);
3. **new samples**: $z \sim \mathcal{N}(0, I)$ from the prior, passed through the decoder.

(About 30 seconds on a CPU.)

In [ ]:
class VAE(nn.Module):
    def __init__(self, z_dim=2, hidden=512):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.fc_mu = nn.Linear(hidden, z_dim)
        self.fc_logvar = nn.Linear(hidden, z_dim)
        self.decoder = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(),
                                     nn.Linear(hidden, 784))

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

def train_vae(beta, epochs=15, batch_size=128, lr=1e-3):
    torch.manual_seed(453)
    model = VAE()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(453)
    for _ in range(epochs):
        idx = torch.randperm(len(X_mnist), generator=g)
        for start in range(0, len(X_mnist), batch_size):
            x = X_mnist[idx[start:start + batch_size]]
            logits, mu, logvar = model(x)
            loss, rec, kl = vae_loss(logits, x, mu, logvar, beta=beta)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model

def montage(images, nrow):
    # images: (n, 784) in [0, 1] -> one (nrow*28, ncol*28) array
    n = images.shape[0]
    ncol = math.ceil(n / nrow)
    grid = torch.ones(nrow * 28, ncol * 28)
    for i in range(n):
        r, c = divmod(i, ncol)
        grid[r * 28:(r + 1) * 28, c * 28:(c + 1) * 28] = images[i].view(28, 28)
    return grid

vaes = {beta: train_vae(beta) for beta in [1.0, 0.0]}

fig, axes = plt.subplots(2, 3, figsize=(15, 9.5), gridspec_kw={'width_ratios': [1.1, 1, 1]})
for row, (beta, model) in enumerate(vaes.items()):
    model.eval()
    with torch.no_grad():
        logits, mu, logvar = model(X_mnist_test)
        _, rec, kl = vae_loss(logits, X_mnist_test, mu, logvar)
        recon = torch.sigmoid(model.decoder(mu[:10]))
        prior_samples = torch.sigmoid(model.decoder(torch.randn(64, 2, generator=torch.Generator().manual_seed(0))))
    sc = axes[row, 0].scatter(mu[:, 0], mu[:, 1], c=y_mnist_test, cmap='tab10', s=3)
    axes[row, 0].add_patch(plt.Circle((0, 0), 2, fill=False, color='k', lw=1.5))
    axes[row, 0].set(title=f'beta = {beta:g}: encoder means mu(x)', xlabel='$z_1$', ylabel='$z_2$', aspect='equal')
    axes[row, 1].imshow(montage(torch.cat([X_mnist_test[:10], recon]), nrow=2), cmap='gray')
    axes[row, 1].set_title(f'beta = {beta:g}: originals (top) / reconstructions')
    axes[row, 2].imshow(montage(prior_samples, nrow=8), cmap='gray')
    axes[row, 2].set_title(f'beta = {beta:g}: samples z ~ N(0, I)')
    for ax in axes[row, 1:]:
        ax.axis('off')
    print(f'beta = {beta:g}: test rec = {rec:.1f}, test KL = {kl:.2f}, '
          f'std of mu over the data = {mu.std(dim=0).mean():.2f}, '
          f'average encoder sigma = {torch.exp(0.5 * logvar).mean():.4f}')
fig.colorbar(sc, ax=axes[:, 0], shrink=0.6, label='digit')
plt.show()

**What to look for** (not graded). Both models reconstruct test digits about equally well, but only the $\beta = 1$ model's prior samples look like a draw from the whole dataset. With $\beta = 0$ nothing ties the codes to the prior. The encoder spreads its means far outside the $\pm 2$ circle and shrinks its noise to almost nothing, so it is effectively a plain autoencoder. Samples $z \sim \mathcal{N}(0, I)$ land only in the small central region (only a few digit classes live there), or in gaps between codes where the decoder was never trained. With $\beta = 1$ the KL term packs every class's codes into the prior's region and keeps $\sigma$ from collapsing. The decoder learns to decode whole neighborhoods, which is what makes the latent space continuous: walking across it morphs one digit smoothly into its neighbor.

### A.4) How tight is the bound? (9 pts)

The ELBO is a *lower* bound on $\log p_\theta(x)$. One way to see how loose it is: for any $q(z \mid x)$,

$$p_\theta(x) = \mathbb{E}_{z \sim q(z\mid x)}\big[w(z)\big], \qquad \log w(z) = \log p_\theta(x \mid z) + \log p(z) - \log q(z \mid x).$$

Averaging $\log w$ over samples $z \sim q$ gives the ELBO. Averaging $w$ itself over $K$ samples and *then* taking the log gives the **importance-weighted** estimate $\log \frac1K \sum_k w(z_k)$. It is a tighter lower bound, and it approaches $\log p_\theta(x)$ as $K \to \infty$.

Complete `log_weight(model, x, z, mu, logvar)`: the log importance weight $\log w(z)$ for given samples `z`, where
- $\log p_\theta(x \mid z)$ is the Bernoulli log-likelihood of `x` under the decoder's logits `model.decoder(z)`, summed over pixels (the negative of the reconstruction term in A.2),
- $\log p(z)$ is the $\mathcal{N}(0, I)$ log-density, summed over latent dimensions,
- $\log q(z \mid x)$ is the $\mathcal{N}(\mu, \operatorname{diag}(\sigma^2))$ log-density with $\sigma^2 = \exp(\texttt{logvar})$, summed over latent dimensions.

Sum over the **last** dimension, so the function also works when `x` and `z` have an extra leading dimension of $K$ samples. Write out the Gaussian log-densities yourself (don't use `torch.distributions`; the self-check does).

In [ ]:
def log_weight(model, x, z, mu, logvar):
    log_px_z = # <your code>
    log_pz = # <your code>
    log_qz = # <your code>
    return log_px_z + log_pz - log_qz

In [ ]:
def reference_log_weight(model, x, z, mu, logvar):
    return (Bernoulli(logits=model.decoder(z), validate_args=False).log_prob(x).sum(-1)
            + Normal(0.0, 1.0).log_prob(z).sum(-1)
            - Normal(mu, torch.exp(0.5 * logvar)).log_prob(z).sum(-1))

torch.manual_seed(0)
model = vaes[1.0]
x = X_mnist_test[:6]
with torch.no_grad():
    mu, logvar = model.encode(x)
    z = mu + 1.3 * torch.randn_like(mu)
    lw = log_weight(model, x, z, mu, logvar)
    assert lw.shape == (6,), "log_weight should return one value per example"
    assert torch.allclose(lw, reference_log_weight(model, x, z, mu, logvar), atol=1e-3), \
        "log_weight doesn't match torch.distributions"
    zK = mu + torch.randn(5, 6, 2)
    lwK = log_weight(model, x.expand(5, 6, 784), zK, mu, logvar)
    assert lwK.shape == (5, 6), "with K samples per example, log_weight should return shape (K, batch)"
    assert torch.allclose(lwK, reference_log_weight(model, x.expand(5, 6, 784), zK, mu, logvar), atol=1e-3)
print('looks good!')

**No modification required.** For 500 test digits, this draws $K = 100$ samples $z \sim q(z \mid x)$ from each trained VAE and compares the ELBO estimate (the average of $\log w$) with the importance-weighted estimate of $\log p_\theta(x)$. All numbers are in nats per image; higher is better.

In [ ]:
K = 100
x = X_mnist_test[:500]
print(f"{'model':10s} {'ELBO':>9s} {'IW estimate (K=100)':>20s} {'gap':>7s}")
for beta, model in vaes.items():
    torch.manual_seed(0)
    with torch.no_grad():
        mu, logvar = model.encode(x)
        z = mu + torch.exp(0.5 * logvar) * torch.randn(K, *mu.shape)
        lw = log_weight(model, x.expand(K, *x.shape), z, mu, logvar)      # (K, batch)
        elbo = lw.mean(dim=0)
        iw = torch.logsumexp(lw, dim=0) - math.log(K)
    print(f'beta = {beta:g}  {elbo.mean():9.1f} {iw.mean():20.1f} {(iw - elbo).mean():7.2f}')

**What to look for** (not graded). The importance-weighted estimate is always at least the ELBO, and the gap is a lower bound on how much the ELBO underestimates $\log p_\theta(x)$. That underestimate equals $\mathrm{KL}\big(q(z\mid x)\,\|\,p_\theta(z\mid x)\big)$: how far the encoder is from the decoder's true posterior. For $\beta = 1$ the ELBO is only a few nats below the importance-weighted estimate. The $\beta = 0$ model's ELBO is about 20 nats worse, mostly because of its large KL term. Its gap looks *smaller*, but that's because its encoder's $\sigma$ is so tiny that all 100 samples land almost on top of each other, so importance sampling can't explore the posterior either: there, both numbers are loose bounds.

## Part B: Diffusion Models (30 pts)

A diffusion model (here DDPM; Ho, Jain & Abbeel, 2020) defines a fixed **forward process** that gradually adds Gaussian noise to a data point $x_0$ over $T$ steps,

$$q(x_t \mid x_{t-1}) = \mathcal{N}\big(\sqrt{\alpha_t}\, x_{t-1},\; \beta_t I\big), \qquad \alpha_t = 1 - \beta_t,$$

until $x_T$ is (almost) pure $\mathcal{N}(0, I)$ noise. It then trains a network $\hat\varepsilon_\theta(x_t, t)$ to **undo** one step at a time. Sampling starts from fresh noise and runs the learned reverse process.

**Indexing convention:** the math above uses $t = 1, \dots, T$. In code, Python index `t = 0, ..., T-1` stands for math step $t+1$, so `alpha_bars[t]` is $\bar\alpha_{t+1}$. You only need to use the arrays consistently: everything below is written in terms of the arrays.

### B.1) Noise schedule and the forward process in closed form (8 pts)

Let $\bar\alpha_t = \prod_{s=1}^{t} \alpha_s$. Running the forward process for $t$ steps has a closed form, so a noisy example at any step takes one line instead of $t$ sequential noising steps:

$$q(x_t \mid x_0) = \mathcal{N}\big(\sqrt{\bar\alpha_t}\, x_0,\; (1-\bar\alpha_t)\, I\big), \qquad\text{i.e.}\qquad x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\;\varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I).$$

Complete the noise schedule and `q_sample()`. Use $T = 100$ and a **linear** schedule: `betas` evenly spaced from `1e-4` to `0.1` (`torch.linspace`). Then compute `alphas` and their cumulative product `alpha_bars` (`torch.cumprod`). `q_sample(x0, t, eps)` takes a batch `x0` of shape `(n, 2)`, a batch of integer timesteps `t` of shape `(n,)`, and noise `eps` of shape `(n, 2)`, and returns $x_t$ using the closed form.

*(Hint: `alpha_bars[t]` has shape `(n,)`. Use `[:, None]` to broadcast it against `(n, 2)`.)*

In [ ]:
T = 100
betas = # <your code>
alphas = # <your code>
alpha_bars = # <your code>

def q_sample(x0, t, eps):
    ab = # <your code>
    return # <your code>

The self-check compares your one-shot `q_sample` against actually running the step-by-step forward chain $x_t = \sqrt{\alpha_t}x_{t-1} + \sqrt{\beta_t}\,\varepsilon_t$ (given below). The two should produce the same distribution.

In [ ]:
def forward_chain(x0, n_steps, generator):
    x = x0.clone()
    for t in range(n_steps):
        x = torch.sqrt(alphas[t]) * x + torch.sqrt(betas[t]) * torch.randn(x.shape, generator=generator)
    return x

assert betas.shape == (T,) and abs(betas[0].item() - 1e-4) < 1e-8 and abs(betas[-1].item() - 0.1) < 1e-6, "check betas"
g = torch.Generator().manual_seed(0)
x0 = torch.tensor([[1.5, -0.5]]).repeat(50000, 1)
for n_steps in [1, 10, 40]:
    chain = forward_chain(x0, n_steps, g)
    t = torch.full((50000,), n_steps - 1, dtype=torch.long)       # n_steps steps done = Python index n_steps-1
    shot = q_sample(x0, t, torch.randn(x0.shape, generator=g))
    assert torch.allclose(chain.mean(0), shot.mean(0), atol=0.02), f"means differ after {n_steps} steps"
    assert torch.allclose(chain.std(0), shot.std(0), atol=0.02), f"stds differ after {n_steps} steps"
assert alpha_bars[-1] < 0.01, "after T steps almost no signal should remain"
print('looks good!')

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
x0 = X_ring[:3000]
for ax, t in zip(axes, [0, 5, 15, 30, 60, 99]):
    xt = q_sample(x0, torch.full((len(x0),), t, dtype=torch.long), torch.randn_like(x0))
    plot_points(ax, xt, f't = {t}   (alpha_bar = {alpha_bars[t]:.3f})')
plt.suptitle('Forward process: the ring dissolves into N(0, I)', y=1.03)
plt.show()

### B.2) The training loss (6 pts)

DDPM trains the network to predict the noise that was added. Complete `diffusion_loss(model, x0)`. For a batch `x0`:

1. draw a timestep `t` uniformly from `{0, ..., T-1}` for each example (`torch.randint`),
2. draw noise `eps` from $\mathcal{N}(0, I)$, the same shape as `x0`,
3. make `xt` with your `q_sample`,
4. return the mean squared error between `model(xt, t)` and `eps`.

In [ ]:
def diffusion_loss(model, x0):
    t = # <your code>
    eps = # <your code>
    xt = # <your code>
    return # <your code>

In [ ]:
class ZeroModel(nn.Module):
    def forward(self, x, t):
        return torch.zeros_like(x)

torch.manual_seed(0)
loss = diffusion_loss(ZeroModel(), X_ring[:20000])
assert loss.dim() == 0, "the loss should be a scalar"
assert abs(loss.item() - 1.0) < 0.03, "predicting 0 should give a loss of about E[eps^2] = 1"

class SpyModel(nn.Module):
    def forward(self, x, t):
        self.t = t
        return torch.zeros_like(x)

spy = SpyModel()
diffusion_loss(spy, X_ring[:20000])
assert spy.t.min() == 0 and spy.t.max() == T - 1 and spy.t.dtype == torch.long, "t should be integers in 0..T-1"
print('looks good!')

**No modification required.** The denoiser is a small MLP that also receives the timestep, through a learned embedding. The training loop calls your `diffusion_loss` (about 10 seconds).

In [ ]:
class Denoiser(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.t_embed = nn.Embedding(T, hidden)
        self.inp = nn.Linear(2, hidden)
        self.net = nn.Sequential(nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 2))

    def forward(self, x, t):
        return self.net(self.inp(x) + self.t_embed(t))

torch.manual_seed(453)
denoiser = Denoiser()
opt = torch.optim.Adam(denoiser.parameters(), lr=1e-3)
g = torch.Generator().manual_seed(453)
losses = []
for step in range(6000):
    x0 = X_ring[torch.randint(0, len(X_ring), (256,), generator=g)]
    loss = diffusion_loss(denoiser, x0)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

plt.figure(figsize=(6, 3))
plt.plot(torch.tensor(losses).view(-1, 50).mean(dim=1))
plt.xlabel('Step (x50)'); plt.ylabel('Noise-prediction MSE')
plt.show()

### B.3) Sampling: running the process backwards (10 pts)

Complete `p_sample_loop()`, the DDPM sampler. Start from $x \sim \mathcal{N}(0, I)$, then for `t = T-1, T-2, ..., 0` compute

$$\mu_t = \frac{1}{\sqrt{\alpha_t}}\left(x - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\;\hat\varepsilon_\theta(x, t)\right), \qquad x \leftarrow \begin{cases}\mu_t + \sqrt{\beta_t}\; z,\;\; z \sim \mathcal{N}(0, I) & \text{if } t > 0\\[2pt] \mu_t & \text{if } t = 0 \text{ (no noise on the last step).}\end{cases}$$

In words: estimate the noise, remove the right fraction of it, rescale, and then add back a little fresh noise, except at the final step. Draw all random numbers with `generator=generator` (for example `torch.randn(n, 2, generator=generator)`) so the results are reproducible. `p_sample_loop` also records a snapshot of `x` after every step, which we'll plot.

In [ ]:
@torch.no_grad()
def p_sample_loop(model, n, generator=None):
    x = torch.randn(n, 2, generator=generator)
    snapshots = {T: x.clone()}
    for t in reversed(range(T)):
        eps_hat = model(x, torch.full((n,), t, dtype=torch.long))
        mean = # <your code>
        if t > 0:
            x = # <your code>
        else:
            x = # <your code>
        snapshots[t] = x.clone()
    return x, snapshots

The self-check runs your sampler with fake "denoisers" that always predict the same constant noise $c$. In that case the mean and variance of the samples can be computed exactly, step by step: $m \leftarrow (m - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}c)/\sqrt{\alpha_t}$ and $v \leftarrow v/\alpha_t + \beta_t$ (without the $+\beta_t$ at $t = 0$). The check compares your sampler against those formulas.

In [ ]:
class ConstModel(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
    def forward(self, x, t):
        return torch.full_like(x, self.c)

for c in [0.0, 0.3]:
    m, v = 0.0, 1.0
    for t in reversed(range(T)):
        m = (m - betas[t] / torch.sqrt(1 - alpha_bars[t]) * c) / torch.sqrt(alphas[t])
        v = v / alphas[t] + (betas[t] if t > 0 else 0.0)
    out, snaps = p_sample_loop(ConstModel(c), 40000, generator=torch.Generator().manual_seed(1))
    assert len(snaps) == T + 1, "there should be a snapshot for every step"
    assert abs(out.mean().item() - float(m)) < 0.02 * math.sqrt(float(v)) + 1e-6, f"sample mean is off (c={c})"
    assert abs(out.var().item() / float(v) - 1) < 0.03, f"sample variance is off (c={c})"
print('looks good!')

In [ ]:
diff_samples, snaps = p_sample_loop(denoiser, 3000, generator=torch.Generator().manual_seed(0))
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, t in zip(axes, [100, 60, 30, 15, 5, 0]):
    plot_points(ax, snaps[t], f'after reaching t = {t}', color='C2')
plt.suptitle('Reverse process: noise becomes data', y=1.03)
plt.show()
print('diffusion samples:', ring_stats(diff_samples))

### B.4) From predicted noise to a clean estimate (6 pts)

Solving the closed form $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$ for $x_0$, and plugging in the denoiser's *predicted* noise $\hat\varepsilon_\theta(x_t, t)$ instead of the true $\varepsilon$, gives the model's estimate of the clean point that $x_t$ came from:

$$\hat x_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\;\hat\varepsilon_\theta(x_t, t)}{\sqrt{\bar\alpha_t}}.$$

Samplers use this estimate directly (for example, to clip or guide samples). Complete `predict_x0(model, xt, t)` (same shapes and indexing as `q_sample`). The self-check gives it a "denoiser" that knows the true noise, so it must recover $x_0$ exactly, and then checks that your trained denoiser's estimates land near the ring.

In [ ]:
def predict_x0(model, xt, t):
    ab = # <your code>
    eps_hat = # <your code>
    return # <your code>

In [ ]:
class OracleModel(nn.Module):
    def __init__(self, eps):
        super().__init__()
        self.eps = eps
    def forward(self, x, t):
        return self.eps

g = torch.Generator().manual_seed(0)
x0 = sample_ring(1000, generator=g)
t = torch.randint(0, T, (1000,), generator=g)
eps = torch.randn(1000, 2, generator=g)
assert torch.allclose(predict_x0(OracleModel(eps), q_sample(x0, t, eps), t), x0, atol=1e-3), \
    "with the true noise, predict_x0 should recover x0 exactly"

t = torch.full((2000,), 10, dtype=torch.long)
xt = q_sample(sample_ring(2000, generator=g), t, torch.randn(2000, 2, generator=g))
with torch.no_grad():
    x0_hat = predict_x0(denoiser, xt, t)
noisy, denoised = ring_stats(xt)['frac_good'], ring_stats(x0_hat)['frac_good']
print(f'fraction of points near a cluster at t = 10: noisy x_t {noisy:.2f} -> predicted x0 {denoised:.2f}')
assert denoised > noisy + 0.1, "the trained denoiser's x0 estimates should land closer to the clusters than x_t does"
print('looks good!')

**No modification required.** Below, we evaluate the trained denoiser on a grid of points at a small noise level and draw the vector field $-\hat\varepsilon_\theta(x, t)/\sqrt{1-\bar\alpha_t}$. (Arrow lengths are normalized; the color shows the magnitude.)

In [ ]:
t_show = 5
grid = torch.stack(torch.meshgrid(torch.linspace(-3, 3, 25), torch.linspace(-3, 3, 25), indexing='xy'), dim=-1).view(-1, 2)
with torch.no_grad():
    field = -denoiser(grid, torch.full((len(grid),), t_show, dtype=torch.long)) / torch.sqrt(1 - alpha_bars[t_show])
mag = field.norm(dim=1, keepdim=True)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(X_ring[:2000, 0], X_ring[:2000, 1], s=2, alpha=0.3, color='gray')
q = ax.quiver(grid[:, 0], grid[:, 1], (field / mag)[:, 0], (field / mag)[:, 1], mag.squeeze().log(), cmap='viridis')
ax.set(aspect='equal', title=f'$-\\hat\\epsilon(x, t)/\\sqrt{{1-\\bar\\alpha_t}}$ at t = {t_show}')
plt.colorbar(q, ax=ax, shrink=0.75, label='log magnitude')
plt.show()

**What to look for** (not graded). Every arrow points toward the nearest of the 8 clusters, and the magnitude shrinks to about zero at the cluster centers. Moving along $-\hat\varepsilon$ moves a point toward its estimated clean version $\hat x_0$. This field is an estimate of the **score** $\nabla_x \log q_t(x)$, the direction in which the noisy data density increases fastest, and each sampling step follows it a little and adds a little noise. A diffusion model is like a VAE with $T$ latent layers whose "encoder" (the noising process) is fixed rather than learned: its latents end up exactly $\mathcal{N}(0, I)$ by construction, so the prior-mismatch problem from A.3 can't happen. The price is sampling cost: $T = 100$ network evaluations per sample here, versus one decoder pass for a VAE.

## Part C: Generative Adversarial Networks (15 pts)

A GAN trains a generator $G(z)$ and a discriminator $D(x) = \sigma(\ell(x))$, where $\ell$ is the discriminator's **logit**, by playing the minimax game (Goodfellow et al., 2014):

$$\min_G \max_D \; \mathbb{E}_{x\sim p_{\text{data}}}\big[\log D(x)\big] + \mathbb{E}_{z\sim p(z)}\big[\log\big(1 - D(G(z))\big)\big].$$

In practice the generator usually minimizes a different loss, $-\log D(G(z))$ (the **non-saturating** loss), instead of the original $\log(1 - D(G(z)))$ (the **saturating** or minimax loss). In this part you'll find out why.

### C.1) GAN losses (7 pts)

Complete the three losses below. Each takes discriminator **logits** and returns a scalar averaged over the batch. Use `F.binary_cross_entropy_with_logits`, which is numerically stable. Recall that $\mathrm{BCE}(\ell, 1) = -\log\sigma(\ell)$ and $\mathrm{BCE}(\ell, 0) = -\log(1 - \sigma(\ell))$.

- `d_loss(real_logits, fake_logits)`: the discriminator **minimizes** the negative of its objective above: $-\mathbb{E}[\log D(x)] - \mathbb{E}[\log(1 - D(G(z)))]$.
- `g_loss_saturating(fake_logits)`: the generator minimizes $\mathbb{E}[\log(1 - D(G(z)))]$.
- `g_loss_nonsaturating(fake_logits)`: the generator minimizes $-\mathbb{E}[\log D(G(z))]$.

In [ ]:
def d_loss(real_logits, fake_logits):
    return # <your code>

def g_loss_saturating(fake_logits):
    return # <your code>

def g_loss_nonsaturating(fake_logits):
    return # <your code>

In [ ]:
real_l, fake_l = torch.tensor([[2.0], [0.5]]), torch.tensor([[-1.0], [0.0]])
D_real, D_fake = torch.sigmoid(real_l), torch.sigmoid(fake_l)
assert torch.allclose(d_loss(real_l, fake_l), -(torch.log(D_real).mean() + torch.log(1 - D_fake).mean()), atol=1e-6)
assert torch.allclose(g_loss_saturating(fake_l), torch.log(1 - D_fake).mean(), atol=1e-6)
assert torch.allclose(g_loss_nonsaturating(fake_l), -torch.log(D_fake).mean(), atol=1e-6)
assert torch.isfinite(g_loss_nonsaturating(torch.tensor([[-200.0]]))), "use the logits version for numerical stability"
print('looks good!')

### C.2) One GAN training step (8 pts)

GANs are trained by alternating: one update of the discriminator, then one update of the generator. Complete `gan_step(G, D, opt_G, opt_D, real, g_loss_fn, z)`, which does exactly one round:

1. Generate a batch of fakes, `fake = G(z)` (given).
2. **Discriminator update:** compute `d_loss` on `D(real)` and `D(fake.detach())`, then zero `opt_D`'s gradients, backpropagate, and step `opt_D`. Detaching the fakes stops this loss from sending gradients into the generator.
3. **Generator update:** compute `g_loss_fn(D(fake))` with the *just-updated* discriminator, then zero `opt_G`'s gradients, backpropagate, and step `opt_G`. Only the generator moves here, even though gradients flow through `D`.
4. Return the two losses (computed before each update) as Python floats: `(loss_D, loss_G)`.

`g_loss_fn` is one of your C.1 generator losses.

In [ ]:
def make_mlp_2d(out_dim):
    return nn.Sequential(nn.Linear(2, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, out_dim))

def gan_step(G, D, opt_G, opt_D, real, g_loss_fn, z):
    fake = G(z)
    # <your code>
    return loss_D.item(), loss_G.item()

The self-check runs your `gan_step` with plain SGD and checks that each network moved by exactly one gradient step on its own loss: the discriminator on `d_loss` with the starting generator, and the generator on `g_loss_fn` with the *updated* discriminator. It also checks the returned losses and that each loss went down. (Both networks start with stale gradients, as they would mid-training.)

In [ ]:
def flat(model):
    return torch.cat([p.detach().reshape(-1) for p in model.parameters()])

def one_sgd_step(model, loss, lr):
    grads = torch.autograd.grad(loss, list(model.parameters()))
    return torch.cat([(p - lr * g).detach().reshape(-1) for p, g in zip(model.parameters(), grads)])

for g_loss_fn in [g_loss_nonsaturating, g_loss_saturating]:
    torch.manual_seed(0)
    G, D = make_mlp_2d(2), make_mlp_2d(1)
    real, z = sample_ring(64), torch.randn(64, 2)
    G0, D0 = copy.deepcopy(G), copy.deepcopy(D)
    lr = 0.05
    opt_G, opt_D = torch.optim.SGD(G.parameters(), lr=lr), torch.optim.SGD(D.parameters(), lr=lr)
    for prm in list(G.parameters()) + list(D.parameters()):    # stale gradients from "earlier steps"
        prm.grad = torch.randn_like(prm)
    out = gan_step(G, D, opt_G, opt_D, real, g_loss_fn, z)

    assert isinstance(out, tuple) and len(out) == 2 and all(isinstance(v, float) for v in out), \
        "return (loss_D, loss_G) as Python floats"
    # Discriminator: one SGD step on d_loss, with fakes from the *starting* generator.
    D_loss0 = d_loss(D0(real), D0(G0(z).detach()))
    assert math.isclose(out[0], D_loss0.item(), rel_tol=1e-5), "loss_D should be d_loss before the discriminator update"
    assert torch.allclose(flat(D), one_sgd_step(D0, D_loss0, lr), atol=1e-6), \
        "D should move by exactly one step on d_loss (and must not move during the generator update)"
    # Generator: one SGD step on g_loss_fn, evaluated with the *updated* discriminator.
    G_loss0 = g_loss_fn(D(G0(z)))
    assert math.isclose(out[1], G_loss0.item(), rel_tol=1e-5), \
        "loss_G should be g_loss_fn(D(fake)) with the updated D, before the generator update"
    assert torch.allclose(flat(G), one_sgd_step(G0, G_loss0, lr), atol=1e-6), \
        "G should move by exactly one step on its own loss (did d_loss gradients leak into G? .detach() and zero_grad)"
    # And both updates should help their own objective on this batch.
    with torch.no_grad():
        assert d_loss(D(real), D(G0(z))) < D_loss0, "the discriminator's loss should go down after its update"
        assert g_loss_fn(D(G(z))) < G_loss0, "the generator's loss should go down after its update"
print('looks good!')

### C.3) Train a GAN and compare

**No modification required.** We train small MLP GANs on the ring, using *your* losses and *your* `gan_step`. To create the situation where the generator loss matters most, the discriminator first gets a **500-step head start** against the untrained generator. We then measure the generator's gradient norm before its first update under each loss, and train both versions for 3,000 steps with Adam. Finally we compare the GAN's samples with your diffusion samples from B.3 (about 30 seconds).

In [ ]:
def train_gan(g_loss_fn, steps=3000, d_head_start=500, batch_size=256, seed=453):
    torch.manual_seed(seed)
    G, D = make_mlp_2d(2), make_mlp_2d(1)
    opt_G = torch.optim.Adam(G.parameters(), lr=1e-3, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=1e-3, betas=(0.5, 0.999))
    g = torch.Generator().manual_seed(seed)
    def real_batch():
        return X_ring[torch.randint(0, len(X_ring), (batch_size,), generator=g)]

    for _ in range(d_head_start):                  # discriminator-only steps
        with torch.no_grad():
            fake = G(torch.randn(batch_size, 2))
        loss = d_loss(D(real_batch()), D(fake))
        opt_D.zero_grad(); loss.backward(); opt_D.step()

    # The generator's gradient before its first update (separate noise, so training is unaffected).
    fake = G(torch.randn(batch_size, 2, generator=torch.Generator().manual_seed(seed + 1)))
    G.zero_grad()
    g_loss_fn(D(fake)).backward()
    first_grad_norm = torch.sqrt(sum((p.grad ** 2).sum() for p in G.parameters())).item()
    first_D_fake = torch.sigmoid(D(fake)).mean().item()

    snapshots = {}
    for step in range(steps + 1):
        gan_step(G, D, opt_G, opt_D, real_batch(), g_loss_fn, torch.randn(batch_size, 2))
        if step in (0, 250, 1000, 3000):
            with torch.no_grad():
                snapshots[step] = G(torch.randn(3000, 2, generator=torch.Generator().manual_seed(0)))
    return G, snapshots, first_grad_norm, first_D_fake

gans = {}
for name, fn in [('non-saturating', g_loss_nonsaturating), ('saturating (minimax)', g_loss_saturating)]:
    G, snapshots, grad_norm, D_fake = train_gan(fn)
    gans[name] = (G, snapshots)
    print(f'{name:22s} first step: mean D(G(z)) = {D_fake:.4f}, ||grad of G|| = {grad_norm:.4f}')

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, (name, (G, snapshots)) in enumerate(gans.items()):
    for col, (step, x) in enumerate(snapshots.items()):
        plot_points(axes[row, col], x, f'{name}\nstep {step}', color='C3')
plt.show()
for name, (G, snapshots) in gans.items():
    print(f'{name:22s} after 3000 steps:', ring_stats(snapshots[3000]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
plot_points(axes[0], X_ring[:3000], 'data')
plot_points(axes[1], diff_samples, f'diffusion ({T} network calls / sample)', color='C2')
plot_points(axes[2], gans['non-saturating'][1][3000], 'GAN, non-saturating (1 call / sample)', color='C3')
plt.show()
for name, x in [('diffusion', diff_samples), ('GAN (non-saturating)', gans['non-saturating'][1][3000])]:
    print(f'{name:22s}', ring_stats(x))

**What to look for** (not graded). After the head start the discriminator rejects the initial fakes with near certainty, so its logit $\ell$ on fakes is very negative and $\sigma(\ell) = D(G(z)) \approx 0$. The derivative of the saturating loss $\log(1-\sigma(\ell))$ with respect to $\ell$ is $-\sigma(\ell) \approx 0$, while the derivative of the non-saturating loss $-\log\sigma(\ell)$ is $-(1-\sigma(\ell)) \approx -1$. Every gradient into the generator passes through this factor, so the saturating generator's first gradient is roughly $1/D(G(z))$ times smaller: it gets almost no signal exactly when it is worst. Adam partly rescues it: dividing each coordinate's step by its recent gradient size makes a uniformly tiny gradient still take steps of size about $\eta$. That rescue is fragile (the direction of a vanishing gradient is mostly noise), and the saturating run ends up worse. Both the GAN and diffusion cover all 8 clusters, but they miss differently. Diffusion leaves a soft halo around each cluster. The GAN draws thin bridges between clusters, because a continuous generator fed connected noise can't produce 8 separate islands. The GAN needs 1 network evaluation per sample; diffusion needs 100.

Attribution: Ben Lengerich